# 01 · Comprensión de los Datos

> **Objetivo:** entender qué tenemos en el dataset *Online Retail II* antes
> de tocar nada. Toda decisión de limpieza y modelado depende de este paso.

## ¿Qué vamos a hacer?

1. Descargar el dataset (si no está) y cargarlo.
2. Inspeccionar dimensiones, tipos de datos y memoria.
3. Identificar variables clave para segmentación (RFM).
4. Detectar problemas iniciales: faltantes, valores extraños, cancelaciones.

## Concepto teórico breve

El **Customer Relationship Management (CRM)** moderno se basa en la idea
de que **no todos los clientes son iguales**. La segmentación busca agrupar
clientes con comportamiento similar para tratarlos de forma diferenciada.

El framework más usado en retail es **RFM**:

- **Recency**: ¿cuándo compró por última vez?
- **Frequency**: ¿con qué frecuencia compra?
- **Monetary**: ¿cuánto gasta?

Para construir RFM necesitamos un dataset **transaccional** (una fila por
operación). *Online Retail II* lo es.


In [ ]:
# Permite importar el paquete src/ desde el notebook
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
import pandas as pd
import numpy as np

from src.data.loader import download_dataset, load_raw_transactions
from src.config import RAW_DATA_FILE

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)


## 1. Descarga del dataset

La función ``download_dataset()`` descarga el ZIP desde UCI y descomprime
``online_retail_II.xlsx`` en ``data/raw/``. Si el archivo ya existe,
no vuelve a descargar.

> **Tip:** la primera ejecución toma ~30s (depende de tu red).


In [ ]:
download_dataset()

## 2. Carga de los datos

El archivo Excel trae dos hojas (un año por hoja). El loader las concatena.

> **Tip:** ``pd.read_excel`` es lento para archivos grandes. Para uso
> productivo, después de la primera lectura conviene guardar como ``parquet``.


In [ ]:
df = load_raw_transactions()
print(f"Shape: {df.shape}")
df.head()


## 3. Tipos y memoria

In [ ]:
df.info(memory_usage='deep')

**Lectura del output:**

- ``Invoice`` y ``StockCode`` son objetos (string) — algunas facturas y
  productos contienen letras.
- ``CustomerID`` es ``float64`` con NaN. Tendremos que convertirlo a
  ``int`` después de eliminar los nulos.
- ``InvoiceDate`` ya viene como ``datetime64`` — no hay que parsearla.

## 4. Estadísticas descriptivas


In [ ]:
df.describe(include='all').T

> **Errores comunes que ya saltan a la vista:**
>
> 1. ``Quantity`` y ``Price`` tienen valores **negativos**. ¿Devoluciones?
>    ¿Errores de captura? Hay que decidir qué hacer con ellos.
> 2. ``Quantity`` máximo es muchísimo mayor que la mediana → outliers
>    o clientes mayoristas.
> 3. ``CustomerID`` tiene faltantes → veremos el porcentaje exacto abajo.

## 5. Faltantes


In [ ]:
missing = df.isna().sum()
missing_pct = (df.isna().mean() * 100).round(2)
pd.DataFrame({"missing": missing, "pct": missing_pct}).sort_values("missing", ascending=False)


**Decisión preliminar:** las filas sin ``CustomerID`` no se pueden
atribuir a un cliente, así que no sirven para RFM. Las eliminaremos
en el notebook 02. *Imputarlas sería inventar clientes*.

## 6. Cancelaciones

Las facturas que empiezan con `C` son cancelaciones (devoluciones de
mercancía). Vamos a contarlas.


In [ ]:
df["Invoice"] = df["Invoice"].astype(str)
n_cancel = df["Invoice"].str.startswith("C").sum()
print(f"Cancelaciones: {n_cancel:,} ({100*n_cancel/len(df):.2f}%)")


## 7. Distribución por país


In [ ]:
df["Country"].value_counts().head(10)


> **Observación:** ~90% de las transacciones son del Reino Unido. Si
> quisiéramos segmentar geográficamente, tendríamos que agrupar el resto
> en categorías ("Europa Occidental", "Otro") para que no dominen los
> clusters por sí solos.

## 8. Rango temporal


In [ ]:
print("Rango de fechas:")
print(f"  Mínimo: {df['InvoiceDate'].min()}")
print(f"  Máximo: {df['InvoiceDate'].max()}")
print(f"  Span:   {(df['InvoiceDate'].max() - df['InvoiceDate'].min()).days} días")


## 9. ¿Cuántos clientes tenemos?


In [ ]:
print(f"Clientes únicos (incluyendo NaN): {df['CustomerID'].nunique(dropna=False):,}")
print(f"Clientes únicos (sin NaN):       {df['CustomerID'].nunique():,}")
print(f"Productos únicos (StockCode):    {df['StockCode'].nunique():,}")
print(f"Facturas únicas:                 {df['Invoice'].nunique():,}")


## Resumen del notebook

| Hallazgo | Implicación |
|---|---|
| ~25% de filas sin ``CustomerID`` | Las eliminaremos. |
| Cancelaciones (~2%) | Las eliminaremos. |
| Valores negativos en `Quantity`/`Price` | Filtrar > 0. |
| Outliers de cantidad | Conservar pero transformar (log). |
| Dataset dominado por UK | Si usamos país, agruparlo. |

---

## Preguntas de Reflexión

1. ¿Por qué no podemos imputar los `CustomerID` faltantes con la moda
   (el cliente más frecuente)?
2. ¿Qué problemas traería *no* eliminar las cancelaciones antes del RFM?
3. Si el negocio te pidiera segmentar **clientes mayoristas** específicamente,
   ¿qué variables nuevas considerarías?
4. ¿Qué pasaría si entrenáramos K-Means usando `Quantity` directamente,
   sin transformar los outliers?

> **Próximo paso:** ``02_data_cleaning.ipynb`` — aplicamos las decisiones
> que tomamos aquí.
